# 06 — Monitoreo del Sistema de Recomendación
## Cross-Selling Recommender | Proyecto Final Henry

**Qué monitorea este notebook:**

| Check | Qué detecta |
|---|---|
| Calidad de datos | Schema, nulos >50%, rangos inválidos |
| Artefactos del modelo | Existencia, tamaño, antigüedad |
| Deriva de distribución | KS-test basket size + frecuencia de productos |
| Métricas offline | Precision/Recall/NDCG/HitRate vs umbrales mínimos |

**Por qué KS-test para detectar deriva:**
Kolmogorov-Smirnov es no paramétrico, no asume ninguna distribución.
Basket size tiene distribución right-skewed (mediana ~10, cola larga),
por lo que tests paramétricos como t-test serían inapropiados.
KS compara CDFs empíricas: H0 = misma distribución.

In [ ]:
# ── 1. Configuración ────────────────────────────────────────────────────────
import sys, logging
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

PROJECT_ROOT = Path().resolve().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.monitoring.monitor import RecommenderMonitor

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    datefmt='%H:%M:%S',
)
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 120})
FIGURES_DIR = PROJECT_ROOT / 'outputs' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
print(f'PROJECT_ROOT: {PROJECT_ROOT}')


---
## 2. Inicializar Monitor

In [ ]:
# ── 2.1 Instanciar RecommenderMonitor ───────────────────────────────────────
monitor = RecommenderMonitor(project_root=PROJECT_ROOT)
print(f'Data dir    : {monitor.data_dir}')
print(f'Models dir  : {monitor.models_dir}')
print(f'Reports dir : {monitor.reports_dir}')
print(f'Baseline    : {monitor.baseline_path}')


---
## 3. Check 1 — Calidad de Datos

In [ ]:
# ── 3.1 Validar schema, nulos y rangos ──────────────────────────────────────
dq = monitor.check_data_quality()
print(f'Estado global: {dq["status"].upper()}')
print(f'Issues encontrados: {len(dq["issues"])}')
for issue in dq['issues']:
    print(f'  ⚠️  {issue}')


In [ ]:
# ── 3.2 Tabla de tasas de nulos por tabla ───────────────────────────────────
rows = []
for tname, tdata in dq['tables'].items():
    if 'null_rates_pct' not in tdata:
        continue
    for col, rate in tdata['null_rates_pct'].items():
        rows.append({'tabla': tname, 'columna': col, 'nulos_pct': rate})

if rows:
    df_nulls = pd.DataFrame(rows)
    df_nulls_pivot = df_nulls.pivot_table(
        index='columna', columns='tabla', values='nulos_pct', aggfunc='first'
    ).fillna('-')
    print('Tasas de nulos (%) — primeras 10 000 filas por tabla:')
    display(df_nulls_pivot)
else:
    print('No hay datos de nulos disponibles (archivos no encontrados).')


---
## 4. Check 2 — Artefactos del Modelo

In [ ]:
# ── 4.1 Verificar modelos entrenados ────────────────────────────────────────
art = monitor.check_model_artifacts()
print(f'Estado global: {art["status"].upper()}')
print()
for fname, fdata in art['artifacts'].items():
    icon = {"ok": "✅", "warning": "⚠️", "error": "❌"}.get(fdata.get("status", "ok"), "❓")
    print(f'{icon} {fname}')
    print(f'   Descripción : {fdata.get("description", "-")}')
    if 'size_kb' in fdata:
        print(f'   Tamaño      : {fdata["size_kb"]:.1f} KB')
        print(f'   Antigüedad  : {fdata["age_days"]} días')
        print(f'   Modificado  : {fdata["last_modified"]}')
    for issue in fdata.get('issues', []):
        print(f'   ⚠️  {issue}')
    print()


---
## 5. Check 3 — Deriva de Distribución (KS-test)

El test KS de dos muestras compara si dos distribuciones son iguales.
- **H0:** Las distribuciones son la misma
- **H1:** Las distribuciones difieren
- **α = 0.05** → rechazamos H0 si p-value < 0.05

Cuando no hay datos de producción independientes se realiza un split 50/50
del baseline para validar que el test funciona correctamente (expectativa: sin deriva).

In [ ]:
# ── 5.1 Ejecutar KS-test de deriva ──────────────────────────────────────────
drift = monitor.check_distribution_drift()  # sin current_df → split 50/50 del baseline
print(f'Estado global: {drift["status"].upper()}')
for issue in drift.get('issues', []):
    print(f'  ⚠️  {issue}')
if not drift.get('issues'):
    print('  Sin deriva detectada.')
print()
if 'tests' in drift:
    for test_name, test_data in drift['tests'].items():
        if 'ks_statistic' not in test_data:
            print(f'{test_name}: {test_data.get("message", "—")}')
            continue
        drift_icon = "⚠️  DERIVA" if test_data["drift_detected"] else "✅ OK"
        print(f'{test_name}:')
        print(f'  {drift_icon}  KS={test_data["ks_statistic"]:.4f}  p={test_data["p_value"]:.4f}')
        if 'baseline_mean' in test_data:
            delta = test_data["delta_mean"]
            print(f'  Media baseline: {test_data["baseline_mean"]:.3f}  →  actual: {test_data["current_mean"]:.3f}  (Δ={delta:+.3f})')
        print()


In [ ]:
# ── 5.2 Visualización de distribución de basket size ────────────────────────
baseline_path = monitor.baseline_path
if baseline_path.exists():
    df_baseline = pd.read_parquet(baseline_path)
    n = min(50_000, len(df_baseline))
    df_s = df_baseline.sample(n=n, random_state=0)
    n_half = n // 2
    df_ref = df_s.iloc[:n_half]
    df_cur = df_s.iloc[n_half:]
    bs_ref = df_ref.groupby('order_id')['product_id'].count().values
    bs_cur = df_cur.groupby('order_id')['product_id'].count().values

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    fig.suptitle('Distribución Basket Size — Referencia vs Actual',
                 fontsize=12, fontweight='bold')

    for ax, data, label, color in zip(
        axes,
        [bs_ref, bs_cur],
        ['Referencia (baseline)', 'Actual (50% split)'],
        ['#4C72B0', '#DD8452'],
    ):
        ax.hist(data, bins=40, color=color, alpha=0.85, edgecolor='white',
                range=(1, 40))
        ax.axvline(np.mean(data), color='red', linestyle='--', linewidth=1.5,
                   label=f'Media: {np.mean(data):.2f}')
        ax.axvline(np.median(data), color='orange', linestyle=':', linewidth=1.5,
                   label=f'Mediana: {np.median(data):.0f}')
        ax.set_xlabel('Basket Size (ítems/orden)')
        ax.set_ylabel('Órdenes')
        ax.set_title(label)
        ax.legend(fontsize=9)

    plt.tight_layout()
    fig.savefig(FIGURES_DIR / '18_monitor_basket_size_drift.png', bbox_inches='tight')
    plt.show()
    print('Guardado: outputs/figures/18_monitor_basket_size_drift.png')
else:
    print('Baseline no disponible — ejecutar notebook 02 para generar df_full.parquet')


---
## 6. Check 4 — Métricas Offline

In [ ]:
# ── 6.1 Verificar métricas vs umbrales mínimos ──────────────────────────────
metrics_check = monitor.check_metrics_report()
print(f'Estado global: {metrics_check["status"].upper()}')
for issue in metrics_check.get('issues', []):
    print(f'  ⚠️  {issue}')
print()
if 'metrics_at_k10' in metrics_check:
    for model_name, model_metrics in metrics_check['metrics_at_k10'].items():
        print(f'Modelo: {model_name}')
        for metric, mdata in model_metrics.items():
            icon = "❌" if mdata["below_threshold"] else "✅"
            print(f'  {icon} {metric}: {mdata["value"]:.4f} (umbral: {mdata["threshold"]})')
        print()


In [ ]:
# ── 6.2 Radar / barchart de métricas vs umbrales ────────────────────────────
report_path = monitor.reports_dir / 'evaluation_results.csv'
if report_path.exists():
    df_eval = pd.read_csv(report_path)
    df_k10 = df_eval[df_eval['K'] == 10].copy()
    metrics_cols = ['Precision@K', 'Recall@K', 'NDCG@K', 'HitRate@K']
    thresholds = {'Precision@K': 0.01, 'Recall@K': 0.01, 'NDCG@K': 0.01, 'HitRate@K': 0.05}

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle('Métricas @ K=10 vs Umbrales Mínimos',
                 fontsize=12, fontweight='bold')
    colors = {'ALS': '#4C72B0', 'FP-Growth': '#DD8452'}

    for ax, model_name in zip(axes, ['ALS', 'FP-Growth']):
        row = df_k10[df_k10['Modelo'] == model_name]
        if row.empty:
            ax.set_visible(False)
            continue
        vals = [float(row[m].iloc[0]) for m in metrics_cols]
        tvals = [thresholds[m] for m in metrics_cols]
        x = range(len(metrics_cols))
        bars = ax.bar(x, vals, color=colors[model_name], alpha=0.85,
                      edgecolor='white', label=model_name)
        ax.plot(x, tvals, 'r--', linewidth=1.5, marker='o',
                markersize=5, label='Umbral mínimo')
        ax.set_xticks(list(x))
        ax.set_xticklabels(metrics_cols, rotation=15, ha='right')
        ax.set_ylabel('Valor')
        ax.set_title(model_name)
        ax.legend(fontsize=9)
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.001,
                    f'{val:.4f}', ha='center', va='bottom', fontsize=9)

    plt.tight_layout()
    fig.savefig(FIGURES_DIR / '19_monitor_metrics_thresholds.png', bbox_inches='tight')
    plt.show()
    print('Guardado: outputs/figures/19_monitor_metrics_thresholds.png')
else:
    print('evaluation_results.csv no encontrado — ejecutar notebook 05')


---
## 7. Reporte Completo

In [ ]:
# ── 7.1 Ejecutar reporte completo y guardar JSON ────────────────────────────
report = monitor.run_full_report()
output_path = monitor.save_report(report)
print(f'Reporte guardado: {output_path}')
print(f'Estado global  : {report["global_status"].upper()}')
print(f'Total issues   : {report["total_issues"]}')


In [ ]:
# ── 7.2 Resumen en consola ───────────────────────────────────────────────────
RecommenderMonitor.print_summary(report)


---
## 8. Dashboard de Estado

In [ ]:
# ── 8.1 Panel de semáforo de checks ─────────────────────────────────────────
section_labels = {
    'data_quality'      : 'Calidad\nde Datos',
    'model_artifacts'   : 'Artefactos\ndel Modelo',
    'distribution_drift': 'Deriva de\nDistribución',
    'metrics'           : 'Métricas\nOffline',
}
status_colors = {'ok': '#2ecc71', 'warning': '#f39c12', 'error': '#e74c3c'}
status_labels_es = {'ok': 'OK', 'warning': 'ADVERTENCIA', 'error': 'ERROR'}

fig, ax = plt.subplots(figsize=(10, 3))
ax.axis('off')
n = len(section_labels)
for i, (key, label) in enumerate(section_labels.items()):
    status = report['sections'][key].get('status', 'ok')
    color  = status_colors.get(status, '#95a5a6')
    x = (i + 0.5) / n
    circle = plt.Circle((x, 0.55), 0.12, color=color, transform=ax.transAxes,
                         clip_on=False, zorder=3)
    ax.add_patch(circle)
    ax.text(x, 0.55, status_labels_es.get(status, status),
            ha='center', va='center', fontsize=9, fontweight='bold',
            color='white', transform=ax.transAxes, zorder=4)
    ax.text(x, 0.12, label, ha='center', va='center', fontsize=10,
            transform=ax.transAxes)

ax.set_title(f'Monitor Dashboard — {report["timestamp"][:10]}',
             fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
fig.savefig(FIGURES_DIR / '20_monitor_dashboard.png', bbox_inches='tight', dpi=150)
plt.show()
print('Guardado: outputs/figures/20_monitor_dashboard.png')


---
## 9. Conclusiones de Monitoreo

### Cuándo reaccionar

| Check | Umbral de acción |
|---|---|
| `data_quality` | Cualquier `error` → detener pipeline |
| `model_artifacts` | `age_days > 90` → reentrenar |
| `distribution_drift` | KS p-value < 0.05 → investigar cambio de comportamiento |
| `metrics` | Métrica < umbral → revisar modelo y datos |

### Próximos pasos de monitoreo (Sprint 3)
- Integrar con MLflow para tracking automático de métricas
- Configurar alertas vía email/Slack cuando `global_status != "ok"`
- Calcular PSI (Population Stability Index) como complemento al KS-test
- A/B testing entre ALS y FP-Growth en producción
